Lesional:
* sub-EXAMPLE,1,right,multifocal,Other,"pathology_type, hemorrhage, periventricular encephalomalacia"
* sub-EXAMPLE,1,bilateral,multifocal,pathology_type,
* sub-EXAMPLE,1,left,temporal,other,abnormal signal; meningoencephalitis sequelae
* sub-EXAMPLE,1,right,temporal,Other,Small focus of encephalomalacia with hemosiderin deposition staining in the lateral anterior right parahippocampal gyrus
* sub-EXAMPLE,1,right,temporal,Other,Right pathology_type vs pathology_type vs low grade tumor s/p partial resection
* sub-EXAMPLE,1,left,temporal,pathology_type,Left pathology_type with volume loss of left hippocampus
* sub-EXAMPLE,1,left,temporal,Other,"Small left anteromesial temporal mass, low grade glioma vs DNET"

Also considered but unsuitable:
* sub-EXAMPLE
* sub-EXAMPLE
* sub-EXAMPLE
* sub-EXAMPLE

Back up:
* sub-EXAMPLE (left temporal multifocal (pathology_type ablation + mesial temporal) good outcome w/ mask) - asymmetries are subtle/inconclusive

Good options for publication:
* sub-EXAMPLE (right pathology_type ATL bad outcome w/ mask) - asymmetries are clearly visible, but not obviously outside resection mask
* sub-EXAMPLE (left temporal ablation good outcome w/ mask) - asymmetries are subtle/inconclusive
* sub-EXAMPLE (right temporal ablation bad outcome w/ mask) - asymmetries are subtle/inconclusive
* sub-EXAMPLE (left temporal ablation good outcome w/ mask) - asymmetries are subtle
* sub-EXAMPLE (left temporal ablation bad outcome w/ mask) - asymmetries are contrast-dependent, much more noticable with ICVF than RTPP

Confirmed!

* sub-EXAMPLE

In [ ]:
from ipyniivue import MultiplanarType, NiiVue
import ipyniivue
import ipywidgets as widgets
from IPython.display import display
import os
import json
import numpy as np
import nibabel as nib
import matplotlib
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.gridspec import GridSpec
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from glob import glob

_MPL_CMAP_CACHE = {}


def _niivue_colormap_json_path(name):
    pkg_dir = os.path.dirname(ipyniivue.__file__)
    return os.path.join(pkg_dir, "static", "colormaps", f"{name.lower()}.json")


def _load_niivue_colormap(name):
    path = _niivue_colormap_json_path(name)
    if not os.path.exists(path):
        return None
    with open(path, "r") as f:
        cm = json.load(f)
    x = np.array(cm["I"], dtype=float) / 255.0
    r = np.array(cm["R"], dtype=float) / 255.0
    g = np.array(cm["G"], dtype=float) / 255.0
    b = np.array(cm["B"], dtype=float) / 255.0
    cdict = {
        "red": [(float(x[i]), float(r[i]), float(r[i])) for i in range(len(x))],
        "green": [(float(x[i]), float(g[i]), float(g[i])) for i in range(len(x))],
        "blue": [(float(x[i]), float(b[i]), float(b[i])) for i in range(len(x))],
    }
    return LinearSegmentedColormap(name, cdict)


def mpl_colormap_for_save(name):
    """Return a matplotlib colormap matching ipyniivue's named colormap."""
    if name in _MPL_CMAP_CACHE:
        return _MPL_CMAP_CACHE[name]
    try:
        matplotlib.colormaps.get_cmap(name)
        _MPL_CMAP_CACHE[name] = name
        return name
    except (KeyError, ValueError):
        pass
    niivue_cmap = _load_niivue_colormap(name)
    if niivue_cmap is not None:
        _MPL_CMAP_CACHE[name] = niivue_cmap
        return niivue_cmap
    _MPL_CMAP_CACHE[name] = "gray"
    return "gray"

# INPUT
sub = "sub-EXAMPLE"

qsiprep_dir = "../../../derivatives/qsiprep/penn_epilepsy"
t1w_file = f"{qsiprep_dir}/{sub}/anat/{sub}_space-ACPC_desc-preproc_T1w.nii.gz"

qsirecon_dir = "../../../derivatives/qsirecon/penn_epilepsy/derivatives"
postop_dir = "../../../derivatives/postop"

scalar_to_directory = f"../../../data/metadata/scalar_labels_to_directories.json"
scalar_to_filename = f"../../../data/metadata/scalar_labels_to_filenames.json"
scalar_to_human = f"../../../data/metadata/scalar_labels_to_human.json"
with open(scalar_to_directory, "r") as f:
    scalar_to_directory_map = json.load(f)
with open(scalar_to_filename, "r") as f:
    scalar_to_filename_map = json.load(f)
with open(scalar_to_human, "r") as f:
    scalar_to_human_map = json.load(f)

EXCLUDED_SCALARS = [
    "map_li", "map_am", "dti_txx", "dti_txy", "dti_txz", 
    "dti_tyy", "dti_tyz", "dti_tzz", "dti_ha", "rdi_rd1", "rdi_rd2", "gqi_iso"
]

DEFAULT_DWI_SCALAR_LABELS = ["map_rtop", "dti_md", "noddi_icvf", "dti_fa"]

# Function to get all scalar paths for a subject, EXCLUDING the scalars above
def get_scalar_paths(qsirecon_dir, sub, scalar_to_directory_map, scalar_to_filename_map):
    scalar_paths = []
    # Loop over all scalar labels (keys are label names)
    for scalar_label, directory in scalar_to_directory_map.items():
        if scalar_label in EXCLUDED_SCALARS:
            continue  # Skip excluded scalars
        filename = scalar_to_filename_map.get(scalar_label)
        if filename is None:
            continue  # skip if no filename for this label
        # Construct directory path for this scalar
        subject_dir = os.path.join(qsirecon_dir, directory, sub)
        if not os.path.exists(subject_dir):
            continue
        # Find all session subdirectories
        for ses_folder in os.listdir(subject_dir):
            ses_dir = os.path.join(subject_dir, ses_folder)
            if not os.path.isdir(ses_dir):
                continue
            dwi_dir = os.path.join(ses_dir, "dwi")
            # Check for dwi directory
            if not os.path.isdir(dwi_dir):
                continue
            # Build the expected filename pattern
            expected_pattern = f"{sub}_{ses_folder}_space-ACPC_{filename}_dwimap.nii.gz"
            full_path = os.path.join(dwi_dir, expected_pattern)
            # Check if the file exists
            if os.path.exists(full_path):
                scalar_paths.append(full_path)
    return scalar_paths

scalar_paths = get_scalar_paths(qsirecon_dir, sub, scalar_to_directory_map, scalar_to_filename_map)


# Check if the file exists
if not os.path.exists(t1w_file):
    raise FileNotFoundError(f"The file {t1w_file} does not exist.")

def single_volume(t1w_file):
    nv = NiiVue()
    nv.load_volumes([{'path': t1w_file}])
    return nv


def get_dwi_scalar_display_options(scalar_paths, scalar_to_filename_map, scalar_to_human_map):
    """Build (display_name, path) list for DWI scalar dropdown from scalar paths."""
    filename_to_label = {v: k for k, v in scalar_to_filename_map.items()}
    options = []
    for path in scalar_paths:
        basename = os.path.basename(path)
        part = basename.split("_dwimap")[0]
        filename = part.split("space-ACPC_")[-1] if "space-ACPC_" in part else part
        label = filename_to_label.get(filename, "unknown")
        human = scalar_to_human_map.get(label, label)
        options.append((human, path))
    return options


def resolve_dwi_paths_by_labels(dwi_scalar_options, scalar_to_filename_map, labels, n_dwi):
    """Pick default DWI paths by scalar label keys, falling back to available options."""
    filename_to_label = {v: k for k, v in scalar_to_filename_map.items()}
    label_to_path = {}
    for _human, path in dwi_scalar_options:
        basename = os.path.basename(path)
        part = basename.split("_dwimap")[0]
        filename = part.split("space-ACPC_")[-1] if "space-ACPC_" in part else part
        label = filename_to_label.get(filename, "unknown")
        if label != "unknown":
            label_to_path[label] = path

    all_paths = [path for _, path in dwi_scalar_options]
    default_paths = []
    for i in range(n_dwi):
        label = labels[i] if i < len(labels) else None
        if label is not None and label in label_to_path:
            default_paths.append(label_to_path[label])
        else:
            default_paths.append(all_paths[min(i, len(all_paths) - 1)])
    return default_paths


def t1w_dwi_multi_scalar_volume(
    t1w_file,
    dwi_scalar_options,
    surgery_mask_path=None,
    n_dwi=4,
    output_png_path=None,
    output_screen_png_path=None,
):
    """
    Plot T1w on top and n_dwi DWI scalar rows below, each with its own dropdown.
    If a surgery mask is provided, its outline (red) is overlaid on every viewer.
    dwi_scalar_options: list of (display_name, path) for the DWI scalar dropdowns.
    surgery_mask_path: optional path to surgery segmentation to outline on all viewers.
    output_png_path: optional path for the quick Save .png button (200 dpi).
    output_screen_png_path: optional path for Save screen (1600 dpi), capturing the
        current crosshair position, scalar selections, and colormap.
    """
    if not dwi_scalar_options:
        raise ValueError("No DWI scalar options available")

    viewer_height = 300
    SURGERY_OUTLINE_WIDTH = 1.0
    T1W_OVERLAY_ALPHA = 0.5
    SAVE_PNG_DPI = 200
    SAVE_SCREEN_DPI = 1600
    ZOOM_MIN = 1.0
    ZOOM_MAX = 5.0
    ZOOM_STEP = 0.1
    T1W_ZOOM_MIN = 1.0
    T1W_ZOOM_MAX = 2.0
    T1W_ZOOM_STEP = 0.05

    def tight_nv_layout(height_px):
        h = f"{height_px}px"
        return widgets.Layout(
            height=h,
            min_height=h,
            max_height=h,
            width="100%",
            margin="0",
            padding="0",
            border="0",
        )

    def apply_tight_layout(nv, height_px):
        nv.layout = tight_nv_layout(height_px)

    nv_t1w = NiiVue(
        height=viewer_height,
        multiplanar_force_render=True,
        multiplanar_pad_pixels=0,
        back_color=(0.0, 0.0, 0.0, 1.0),
    )
    nv_dwi_list = [
        NiiVue(
            height=viewer_height,
            multiplanar_force_render=True,
            multiplanar_pad_pixels=0,
            back_color=(0.0, 0.0, 0.0, 1.0),
        )
        for _ in range(n_dwi)
    ]
    all_nvs = [nv_t1w] + nv_dwi_list
    for nv in all_nvs:
        apply_tight_layout(nv, viewer_height)

    has_surgery = surgery_mask_path is not None and os.path.exists(surgery_mask_path)

    default_outline_opacity = 1.0
    default_show_outline = False

    outline_opacity_slider = (
        widgets.FloatSlider(
            min=0.0,
            max=1.0,
            value=default_outline_opacity,
            step=0.05,
            description="Outline opacity:",
            continuous_update=True,
            style={"description_width": "120px"},
            layout=widgets.Layout(width="320px", min_width="280px"),
        )
        if has_surgery
        else None
    )

    show_outline_checkbox = (
        widgets.Checkbox(
            value=default_show_outline,
            description="Show surgery mask outline",
            tooltip="Show or hide the surgery mask outline on all viewers",
        )
        if has_surgery
        else None
    )

    default_show_overlay = True
    show_overlay_checkbox = (
        widgets.Checkbox(
            value=default_show_overlay,
            description="Show surgical overlay (T1w)",
            tooltip=(
                f"Overlay the surgery mask in red at alpha={T1W_OVERLAY_ALPHA:g} "
                "on the T1w viewer only"
            ),
        )
        if has_surgery
        else None
    )

    def current_outline_width():
        if not has_surgery:
            return 0.0
        if show_outline_checkbox is not None and not show_outline_checkbox.value:
            return 0.0
        return SURGERY_OUTLINE_WIDTH

    def current_outline_opacity():
        if not has_surgery:
            return 0.0
        return (
            float(outline_opacity_slider.value)
            if outline_opacity_slider is not None
            else default_outline_opacity
        )

    def surgery_mask_volume_opts():
        return {
            "path": surgery_mask_path,
            "colormap": "red",
            "opacity": current_outline_opacity(),
        }

    if has_surgery:
        for nv in all_nvs:
            nv.overlay_outline_width = current_outline_width()
            nv.overlay_alpha_shader = 0.0

    if has_surgery:
        nv_t1w.load_volumes([{"path": t1w_file}, surgery_mask_volume_opts()])
    else:
        nv_t1w.load_volumes([{"path": t1w_file}])

    default_paths = resolve_dwi_paths_by_labels(
        dwi_scalar_options, scalar_to_filename_map, DEFAULT_DWI_SCALAR_LABELS, n_dwi
    )
    default_cmap = "nih"

    def load_dwi_volume(nv_dwi, path, cmap):
        if has_surgery:
            nv_dwi.load_volumes([
                {"path": path, "colormap": cmap},
                surgery_mask_volume_opts(),
            ])
        else:
            nv_dwi.load_volumes([{"path": path, "colormap": cmap}])

    for nv_dwi, path in zip(nv_dwi_list, default_paths):
        load_dwi_volume(nv_dwi, path, default_cmap)

    cmap_names = nv_dwi_list[0].colormaps()
    colormap_dropdown = widgets.Dropdown(
        options=cmap_names if cmap_names else [default_cmap],
        value=default_cmap if default_cmap in (cmap_names or [default_cmap]) else (cmap_names[0] if cmap_names else default_cmap),
        description="Colormap:",
    )

    dwi_dropdowns = []
    for i in range(n_dwi):
        dwi_dropdowns.append(
            widgets.Dropdown(
                options=[(name, path) for name, path in dwi_scalar_options],
                value=default_paths[i],
                description=f"DWI scalar {i + 1}:",
            )
        )

    def make_dwi_scalar_handler(nv_dwi):
        def on_dwi_scalar_change(change):
            load_dwi_volume(nv_dwi, change["new"], colormap_dropdown.value)
        return on_dwi_scalar_change

    def on_colormap_change(change):
        cmap = change["new"]
        for nv_dwi in nv_dwi_list:
            if len(nv_dwi.volumes) > 0:
                nv_dwi.set_colormap(nv_dwi.volumes[0].id, cmap)

    for dropdown, nv_dwi in zip(dwi_dropdowns, nv_dwi_list):
        dropdown.observe(make_dwi_scalar_handler(nv_dwi), names="value")
    colormap_dropdown.observe(on_colormap_change, names="value")

    layout_dropdown = widgets.Dropdown(
        options=[
            ("Auto", MultiplanarType.AUTO),
            ("Column", MultiplanarType.COLUMN),
            ("Grid", MultiplanarType.GRID),
            ("Row", MultiplanarType.ROW),
        ],
        value=MultiplanarType.AUTO,
        description="Layout:",
    )

    sync_dropdown = widgets.Dropdown(
        options=[
            ("Sync Disabled", 0),
            ("Sync 2D", 1),
            ("Sync 3D", 2),
            ("Sync 2D and 3D", 3),
        ],
        value=3,
        description="Broadcast:",
    )

    crosshair_gap_open = 15.0
    crosshair_open_toggle = widgets.Checkbox(
        value=False,
        description="Open crosshair",
        tooltip="Leave a gap at the crosshair center to see what's underneath",
    )

    crosshair_visible_width = 1.0
    hide_crosshair_toggle = widgets.Checkbox(
        value=False,
        description="Hide crosshair",
        tooltip="Hide the crosshair lines on all viewers",
    )

    zoom_slider = widgets.FloatSlider(
        min=ZOOM_MIN,
        max=ZOOM_MAX,
        value=ZOOM_MIN,
        step=ZOOM_STEP,
        description="Zoom (\u00d7 on bottom-center):",
        continuous_update=True,
        readout_format=".1f",
        style={"description_width": "190px"},
        layout=widgets.Layout(width="460px", min_width="420px"),
    )
    reset_zoom_button = widgets.Button(
        description="Reset zoom",
        tooltip="Reset zoom and pan to default on all viewers",
        layout=widgets.Layout(width="120px"),
    )
    t1w_zoom_slider = widgets.FloatSlider(
        min=T1W_ZOOM_MIN,
        max=T1W_ZOOM_MAX,
        value=T1W_ZOOM_MIN,
        step=T1W_ZOOM_STEP,
        description="Row 1 T1w zoom:",
        continuous_update=True,
        readout_format=".2f",
        style={"description_width": "120px"},
        layout=widgets.Layout(width="320px", min_width="300px"),
    )
    reset_t1w_zoom_button = widgets.Button(
        description="Reset T1w zoom",
        tooltip="Reset only the Row 1 T1w zoom tweak",
        layout=widgets.Layout(width="140px"),
    )
    save_png_button = widgets.Button(
        description="Save .png",
        tooltip=(
            f"Save a quick static render ({SAVE_PNG_DPI} dpi) to {output_png_path}"
            if output_png_path
            else f"Save a quick static render ({SAVE_PNG_DPI} dpi)"
        ),
        disabled=(output_png_path is None),
        layout=widgets.Layout(width="120px"),
    )
    save_screen_button = widgets.Button(
        description=f"Save screen ({SAVE_SCREEN_DPI} dpi)",
        tooltip=(
            f"Save current view ({SAVE_SCREEN_DPI} dpi) to {output_screen_png_path}"
            if output_screen_png_path
            else f"Save current view at {SAVE_SCREEN_DPI} dpi"
        ),
        disabled=(output_screen_png_path is None),
        layout=widgets.Layout(width="180px"),
    )

    status_row = widgets.HTML(
        value="&nbsp;",
        layout=widgets.Layout(margin="0", padding="0", height="16px"),
    )

    # Updated live by ipyniivue when the user moves the crosshair (scene.crosshair_pos
    # in Python is often stale and does not reflect the current slice).
    current_crosshair = {"mm": None, "vox": None, "frac": None}

    def on_crosshair_open_change(change):
        gap = crosshair_gap_open if change["new"] else 0.0
        for nv in all_nvs:
            nv.opts.crosshair_gap = gap

    def on_hide_crosshair_change(change):
        width = 0.0 if change["new"] else crosshair_visible_width
        for nv in all_nvs:
            nv.opts.crosshair_width = width

    def on_layout_change(change):
        new_layout = change["new"]
        for nv in all_nvs:
            nv.opts.multiplanar_layout = new_layout

    def on_sync_change(change):
        v = change["new"]
        is_2d = (v % 2) == 1
        is_3d = v > 1
        for nv in all_nvs:
            others = [other for other in all_nvs if other is not nv]
            nv.broadcast_to(others, {"2d": is_2d, "3d": is_3d})

    def attach_location_handler(nv):
        @nv.on_location_change
        def handle_location(data):
            status_row.value = f"&nbsp;&nbsp;{data['string']}"
            if nv is nv_t1w:
                current_crosshair["mm"] = [float(v) for v in data["mm"][:3]]
                current_crosshair["vox"] = [int(v) for v in data["vox"][:3]]
                current_crosshair["frac"] = [float(v) for v in data["frac"][:3]]

    for nv in all_nvs:
        attach_location_handler(nv)

    def on_outline_opacity_change(change):
        if not has_surgery:
            return
        val = float(change["new"])
        for nv in all_nvs:
            if len(nv.volumes) > 1:
                nv.set_opacity(1, val)

    def on_show_outline_change(change):
        if not has_surgery:
            return
        width = current_outline_width()
        for nv in all_nvs:
            nv.overlay_outline_width = width

    def apply_t1w_overlay():
        if not has_surgery:
            return
        show = show_overlay_checkbox is not None and show_overlay_checkbox.value
        nv_t1w.overlay_alpha_shader = T1W_OVERLAY_ALPHA if show else 0.0

    def on_show_overlay_change(change):
        apply_t1w_overlay()

    DEFAULT_HALF_FOV_MM = 80.0

    def get_half_fov_mm(nv):
        """Return half the volume's mm extent along Y and Z (posterior and inferior)."""
        try:
            mn, mx, _ = nv.scene_extents_min_max()
            half_y = (float(mx[1]) - float(mn[1])) / 2.0
            half_z = (float(mx[2]) - float(mn[2])) / 2.0
            if half_y <= 0:
                half_y = DEFAULT_HALF_FOV_MM
            if half_z <= 0:
                half_z = DEFAULT_HALF_FOV_MM
            return half_y, half_z
        except Exception:
            return DEFAULT_HALF_FOV_MM, DEFAULT_HALF_FOV_MM

    def apply_viewer_zoom_bottom_center(nv, zoom_factor):
        """Zoom one viewer anchored on the bottom-center of each panel.

        Bottom of an axial panel is posterior (-Y); bottom of sagittal/coronal
        panels is inferior (-Z). Setting pan to the world point that should
        appear at the screen center, we anchor the bottom by shifting pan
        toward -Y and -Z by half_fov * (1 - 1/zoom).
        """
        z = float(zoom_factor)
        if z <= 1.0:
            nv.scene.pan2d_xyzmm = [0.0, 0.0, 0.0, 1.0]
            return
        bias = 1.0 - 1.0 / z
        half_y, half_z = get_half_fov_mm(nv)
        nv.scene.pan2d_xyzmm = [0.0, -half_y * bias, -half_z * bias, z]

    def apply_zoom_bottom_center(zoom_factor):
        global_zoom = float(zoom_factor)
        apply_viewer_zoom_bottom_center(nv_t1w, global_zoom * float(t1w_zoom_slider.value))
        for nv in nv_dwi_list:
            apply_viewer_zoom_bottom_center(nv, global_zoom)

    def apply_t1w_zoom():
        apply_viewer_zoom_bottom_center(nv_t1w, float(zoom_slider.value) * float(t1w_zoom_slider.value))

    def on_zoom_change(change):
        apply_zoom_bottom_center(change["new"])

    def on_t1w_zoom_change(change):
        apply_t1w_zoom()

    def on_reset_zoom_click(_):
        changed = False
        if zoom_slider.value != ZOOM_MIN:
            zoom_slider.value = ZOOM_MIN
            changed = True
        if t1w_zoom_slider.value != T1W_ZOOM_MIN:
            t1w_zoom_slider.value = T1W_ZOOM_MIN
            changed = True
        if not changed:
            apply_zoom_bottom_center(ZOOM_MIN)

    def on_reset_t1w_zoom_click(_):
        if t1w_zoom_slider.value != T1W_ZOOM_MIN:
            t1w_zoom_slider.value = T1W_ZOOM_MIN
        else:
            apply_t1w_zoom()

    def _dropdown_label_for_value(dd):
        for label, val in dd.options:
            if val == dd.value:
                return label
        return os.path.basename(str(dd.value))

    def _crosshair_mm():
        if current_crosshair["mm"] is not None:
            return current_crosshair["mm"]
        try:
            frac = list(nv_t1w.scene.crosshair_pos)
            return [float(v) for v in nv_t1w.frac2mm(frac)[:3]]
        except Exception:
            return None

    def _voxel_indices_for_nifti(mm, nii_path, shape):
        img = nib.load(nii_path)
        ijk = np.linalg.inv(img.affine) @ np.array([mm[0], mm[1], mm[2], 1.0], dtype=float)
        dims = shape[:3]
        idx = [int(round(v)) for v in ijk[:3]]
        return tuple(min(max(i, 0), d - 1) for i, d in zip(idx, dims))

    def _voxel_indices_for_save(nii_path, shape):
        crosshair_mm = _crosshair_mm()
        if crosshair_mm is None and current_crosshair["vox"] is None:
            raise ValueError("Crosshair location unknown; click in a viewer first.")

        if current_crosshair["vox"] is not None:
            t1w_shape = nib.load(t1w_file).shape[:3]
            if shape[:3] == t1w_shape and nii_path in (t1w_file, surgery_mask_path):
                vox = current_crosshair["vox"]
                return tuple(
                    min(max(int(v), 0), d - 1) for v, d in zip(vox, shape[:3])
                )

        return _voxel_indices_for_nifti(crosshair_mm, nii_path, shape)

    def _prepare_slice(volume, x_idx, y_idx, z_idx, view_idx):
        if view_idx == 0:
            return np.fliplr(volume[x_idx, :, :].T)
        if view_idx == 1:
            return volume[:, y_idx, :].T
        return volume[:, :, z_idx].T

    def _row_display_range(data):
        finite = data[np.isfinite(data)]
        if finite.size == 0:
            return 0.0, 1.0
        nonzero = finite[finite != 0]
        sample = nonzero if nonzero.size > 0 else finite
        vmin, vmax = np.percentile(sample, [2, 98])
        if vmin >= vmax:
            vmin, vmax = float(np.min(sample)), float(np.max(sample))
        if vmin >= vmax:
            vmax = vmin + 1.0
        return float(vmin), float(vmax)

    def _style_colorbar(cbar):
        cbar.ax.yaxis.set_tick_params(color="white", labelcolor="white")
        cbar.outline.set_edgecolor("white")
        for label in cbar.ax.get_yticklabels():
            label.set_fontfamily("Georgia")

    def _render_static_png(output_path, dpi):
        matplotlib.rcParams.update({
            "font.family": "serif",
            "font.serif": ["Georgia", "DejaVu Serif"],
        })
        if _crosshair_mm() is None and current_crosshair["vox"] is None:
            raise ValueError("Crosshair location unknown; click in a viewer first.")
        show_outline = (
            not has_surgery
            or show_outline_checkbox is None
            or show_outline_checkbox.value
        )
        show_t1w_overlay = (
            has_surgery
            and show_overlay_checkbox is not None
            and show_overlay_checkbox.value
        )
        outline_alpha = (
            float(outline_opacity_slider.value)
            if outline_opacity_slider is not None
            else 1.0
        )

        rows = []
        rows.append({
            "path": t1w_file,
            "data": np.asarray(nib.load(t1w_file).dataobj),
            "label": "T1w",
            "cmap": "gray",
            "is_dwi": False,
        })
        dwi_cmap = mpl_colormap_for_save(colormap_dropdown.value)
        for dd in dwi_dropdowns:
            rows.append({
                "path": dd.value,
                "data": np.asarray(nib.load(dd.value).dataobj),
                "label": _dropdown_label_for_value(dd),
                "cmap": dwi_cmap,
                "is_dwi": True,
            })

        mask_data = None
        if has_surgery:
            mask_data = np.asarray(nib.load(surgery_mask_path).dataobj)

        n_rows = len(rows)
        fig = Figure(figsize=(13, 4 * n_rows), dpi=dpi, facecolor="black")
        FigureCanvasAgg(fig)
        gs = GridSpec(
            n_rows,
            4,
            figure=fig,
            width_ratios=[1, 1, 1, 0.06],
            wspace=0.03,
            hspace=0.05,
        )

        view_names = ["Sagittal", "Coronal", "Axial"]

        for row_idx, row in enumerate(rows):
            data = row["data"]
            x_idx, y_idx, z_idx = _voxel_indices_for_save(row["path"], data.shape)
            row_vmin, row_vmax = _row_display_range(data) if row["is_dwi"] else (None, None)

            mask_slices = None
            if mask_data is not None:
                mxi, myi, mzi = _voxel_indices_for_save(surgery_mask_path, mask_data.shape)
                mask_slices = [
                    _prepare_slice(mask_data, mxi, myi, mzi, view_idx)
                    for view_idx in range(3)
                ]

            im_for_cbar = None
            for col_idx in range(3):
                ax = fig.add_subplot(gs[row_idx, col_idx])
                slc = _prepare_slice(data, x_idx, y_idx, z_idx, col_idx)
                imshow_kwargs = {
                    "X": slc,
                    "cmap": row["cmap"],
                    "origin": "lower",
                    "aspect": "equal",
                }
                if row["is_dwi"]:
                    imshow_kwargs["vmin"] = row_vmin
                    imshow_kwargs["vmax"] = row_vmax
                im = ax.imshow(**imshow_kwargs)
                if row["is_dwi"]:
                    im_for_cbar = im
                if mask_slices is not None:
                    mslc = mask_slices[col_idx]
                    if show_t1w_overlay and row_idx == 0 and np.any(mslc > 0):
                        overlay = np.ma.masked_where(mslc <= 0, mslc)
                        ax.imshow(
                            overlay,
                            cmap="Reds",
                            origin="lower",
                            aspect="equal",
                            alpha=T1W_OVERLAY_ALPHA * outline_alpha,
                            vmin=0,
                            vmax=1,
                        )
                    if show_outline and mslc.size and np.any(mslc > 0) and np.any(mslc == 0):
                        try:
                            ax.contour(
                                mslc,
                                levels=[0.5],
                                colors="red",
                                linewidths=SURGERY_OUTLINE_WIDTH,
                                alpha=outline_alpha,
                            )
                        except Exception:
                            pass
                ax.set_xticks([])
                ax.set_yticks([])
                ax.set_facecolor("black")
                for spine in ax.spines.values():
                    spine.set_visible(False)
                if row_idx == 0:
                    ax.set_title(
                        view_names[col_idx],
                        color="white",
                        fontsize=12,
                        fontfamily="Georgia",
                    )
                if col_idx == 0:
                    ax.set_ylabel(
                        row["label"],
                        color="white",
                        fontsize=11,
                        fontfamily="Georgia",
                    )

            if row["is_dwi"] and im_for_cbar is not None:
                cax = fig.add_subplot(gs[row_idx, 3])
                cbar = fig.colorbar(im_for_cbar, cax=cax)
                _style_colorbar(cbar)

        fig.subplots_adjust(left=0.06, right=0.98, top=0.97, bottom=0.02)
        fig.savefig(output_path, dpi=dpi, facecolor="black", bbox_inches="tight")

    def _run_save(button, output_path, dpi, missing_path_message):
        if not output_path:
            status_row.value = f"&nbsp;&nbsp;{missing_path_message}"
            return
        button.disabled = True
        prev_desc = button.description
        button.description = "Saving..."
        try:
            out_dir = os.path.dirname(output_path)
            if out_dir:
                os.makedirs(out_dir, exist_ok=True)
            _render_static_png(output_path, dpi)
            status_row.value = f"&nbsp;&nbsp;Saved {output_path} ({dpi} dpi)"
        except Exception as exc:
            status_row.value = f"&nbsp;&nbsp;Save failed: {exc}"
        finally:
            button.description = prev_desc
            button.disabled = output_path is None

    def on_save_png_click(_):
        _run_save(
            save_png_button,
            output_png_path,
            SAVE_PNG_DPI,
            "Save .png: no output path configured",
        )

    def on_save_screen_click(_):
        _run_save(
            save_screen_button,
            output_screen_png_path,
            SAVE_SCREEN_DPI,
            "Save screen: no output path configured",
        )

    crosshair_open_toggle.observe(on_crosshair_open_change, names="value")
    hide_crosshair_toggle.observe(on_hide_crosshair_change, names="value")
    layout_dropdown.observe(on_layout_change, names="value")
    sync_dropdown.observe(on_sync_change, names="value")
    if outline_opacity_slider is not None:
        outline_opacity_slider.observe(on_outline_opacity_change, names="value")
    if show_outline_checkbox is not None:
        show_outline_checkbox.observe(on_show_outline_change, names="value")
    if show_overlay_checkbox is not None:
        show_overlay_checkbox.observe(on_show_overlay_change, names="value")
    zoom_slider.observe(on_zoom_change, names="value")
    t1w_zoom_slider.observe(on_t1w_zoom_change, names="value")
    reset_zoom_button.on_click(on_reset_zoom_click)
    reset_t1w_zoom_button.on_click(on_reset_t1w_zoom_click)
    save_png_button.on_click(on_save_png_click)
    save_screen_button.on_click(on_save_screen_click)
    on_sync_change({"new": sync_dropdown.value})
    apply_t1w_overlay()

    second_row_items = [crosshair_open_toggle, hide_crosshair_toggle]
    if show_overlay_checkbox is not None:
        second_row_items.insert(0, show_overlay_checkbox)
    if show_outline_checkbox is not None:
        second_row_items.insert(0, show_outline_checkbox)
    if outline_opacity_slider is not None:
        second_row_items.insert(0, outline_opacity_slider)

    third_row_items = [
        zoom_slider,
        reset_zoom_button,
        t1w_zoom_slider,
        reset_t1w_zoom_button,
        save_png_button,
        save_screen_button,
    ]

    controls = widgets.VBox(
        [
            widgets.HBox(dwi_dropdowns + [colormap_dropdown, layout_dropdown, sync_dropdown]),
            widgets.HBox(second_row_items),
            widgets.HBox(third_row_items),
        ],
        layout=widgets.Layout(margin="0", padding="0"),
    )

    viewers_grid = widgets.GridspecLayout(
        len(all_nvs),
        1,
        grid_gap="0px",
        height=f"{len(all_nvs) * viewer_height}px",
    )
    viewers_grid.layout.margin = "0"
    viewers_grid.layout.padding = "0"
    for i, nv in enumerate(all_nvs):
        viewers_grid[i, 0] = nv

    display(
        widgets.VBox(
            [controls, viewers_grid, status_row],
            layout=widgets.Layout(margin="0", padding="0"),
        )
    )



In [4]:
# T1w (row 1) + 4 DWI scalars (rows 2–5), each with its own dropdown and synced crosshairs
dwi_options = get_dwi_scalar_display_options(
    scalar_paths, scalar_to_filename_map, scalar_to_human_map
)
surgery_mask_path = os.path.join(postop_dir, sub, f"{sub}_ses-postop01_space-ACPC_surgery-segment.nii.gz")
output_png_path = f"../../../derivatives/analysis/intervention_scalar_viz/{sub}_intervention_scalar_viz.png"
output_screen_png_path = f"../../../derivatives/analysis/intervention_scalar_viz/{sub}_intervention_scalar_viz_screen_1600dpi.png"
t1w_dwi_multi_scalar_volume(
    t1w_file,
    dwi_options,
    surgery_mask_path=surgery_mask_path,
    output_png_path=output_png_path,
    output_screen_png_path=output_screen_png_path,
)